In [1]:
import json

# Wczytanie JSON-a z pliku
with open('Seterra (Old Version) - Speedrun.com.json', 'r') as f:
    data = json.load(f)

# Założenie: data to lista
ilosc_elementow = len(data)

print(f"Liczba elementów: {ilosc_elementow}")


Liczba elementów: 1697


In [4]:
def explore_json(obj, path=""):
    if isinstance(obj, dict):
        for k, v in obj.items():
            explore_json(v, f"{path}.{k}" if path else k)
    elif isinstance(obj, list):
        for i, item in enumerate(obj):
            explore_json(item, f"{path}[{i}]")
    else:
        print(f"{path} ({type(obj).__name__}): {repr(obj)}")

explore_json(data)


[0].id (str): 'jp4wyryj'
[0].name (str): 'benken'
[0].url (str): 'benken'
[0].powerLevel (int): 1
[0].color1Id (str): '4e3w46z5'
[0].color2Id (str): '1x3216j7'
[0].colorAnimate (int): 0
[0].areaId (str): 'no'
[1].id (str): 'jo3ko0lj'
[1].name (str): 'Lezt'
[1].url (str): 'Lezt'
[1].powerLevel (int): 1
[1].color1Id (str): 'w461onv1'
[1].color2Id (str): 'gw3r7nq7'
[1].colorAnimate (int): 0
[1].areaId (str): 'no'
[2].id (str): '8wk9r2v8'
[2].name (str): 'unfried'
[2].url (str): 'unfried'
[2].powerLevel (int): 1
[2].color1Id (str): 'w461onv1'
[2].color2Id (str): 'kr3q9nwx'
[2].colorAnimate (int): 0
[2].areaId (str): 'rs'
[3].id (str): 'pj0yzd3x'
[3].name (str): 'Paddi'
[3].url (str): 'Paddi'
[3].powerLevel (int): 1
[3].color1Id (str): '4e3w46z5'
[3].color2Id (str): '4e3w46z5'
[3].colorAnimate (int): 0
[3].areaId (str): 'se/m'
[4].id (str): '18vvw4y8'
[4].name (str): 'Mackan'
[4].url (str): 'Mackan'
[4].powerLevel (int): 0
[4].color1Id (str): 'gy3l7n4l'
[4].color2Id (str): 'gy3l7n4l'
[4].co

In [3]:
import requests
from JsonExtractor import *

gameName = "games"
with open(f'JSON_EXPERIMENTAL_{gameName.upper()}.json', 'w', encoding="utf-8") as f:
    html = requests.get(f"https://www.speedrun.com/pl-PL/{gameName}").text
    hiddenDataJson = JsonExtractor(html).extractHiddenData()
    f.write(json.dumps(hiddenDataJson))

In [ ]:
import requests
from JsonHandler import *
from JsonExtractor import *
import os
from bigtree import Node, tree_to_dot
from functools import partial

def explore_json(obj, path=""):
    if isinstance(obj, dict):
        for k, v in obj.items():
            explore_json(v, f"{path}.{k}" if path else k)
    elif isinstance(obj, list):
        for i, item in enumerate(obj):
            explore_json(item, f"{path}[{i}]")
    else:
        print(f"{path} ({type(obj).__name__}): {repr(obj)}")

def getUrlsForGivenPage(page: int, sortType: str = "name") -> list:
    url = f"https://www.speedrun.com/pl-PL/games?page={page}&platform=&sort={sortType}"
    response = requests.get(url)
    html_text = response.text
    jsonData = JsonExtractor.extractHiddenData(html_text)
    gameUrls = JsonHandler.extractGameUrls(jsonData)
    return gameUrls

def getAllGameUrlsForGivenPages(pageStart: int, pageEnd: int) -> list:
    mergedList = []
    for i in range(pageStart, pageEnd + 1):
        gameUrls = getUrlsForGivenPage(i, "mostruns")
        mergedList.extend(gameUrls)
    return mergedList

def saveJsonDataForGivenGameUrls(gameUrls: list):
    base_url = "https://www.speedrun.com/pl-PL/"
    for game in gameUrls:
        response = requests.get(base_url + game)
        print(f"{game} - response: {response}")
        jsonData = JsonExtractor.extractHiddenData(response.text)
        open(f"experimental_data/{game.upper()}.json", 'w', encoding='utf-8').write(json.dumps(jsonData))


# urls = getAllGameUrlsForGivenPages(1, 2)
# saveJsonDataForGivenGameUrls(urls)



def containsValue(list, key, value):
    return any(elem.get(key) == value for elem in list)

def getAllLinks(jsonDict):
    def filterAnyByIsArchivedFalse(any):
        return not any["archived"]
    def filterCategoriesByPerLevelTrue(category):
        return category["isPerLevel"]
    def filterCategoriesByPerLevelFalse(category):
        return not category["isPerLevel"]
    def filterVariablesForGivenLevelAndCategory(levelId, categoryId, variable):
        categoryScope = variable["categoryScope"]
        levelScope = variable["levelScope"]
        if not variable["isSubcategory"]:
            return False
        if categoryScope == -1 or (categoryScope == 1 and variable["categoryId"] == categoryId):
            if levelScope == 0:
                return False
            else:
                if levelScope == 1:
                    if variable["levelId"] == levelId:
                        return True
                    else:
                        return False
                else:
                    return True
        else:
            return False
    def filterValuesForGivenVariable(variableId, value):
        if variableId == value["variableId"]:
            return True
        else:
            return False

    categories = list(filter(filterAnyByIsArchivedFalse, jsonDict["props"]["pageProps"]["gameData"]["categories"]))
    levels = list(filter(filterAnyByIsArchivedFalse, jsonDict["props"]["pageProps"]["gameData"]["levels"]))
    variables = list(filter(filterAnyByIsArchivedFalse, jsonDict["props"]["pageProps"]["gameData"]["variables"]))
    values = list(filter(filterAnyByIsArchivedFalse, jsonDict["props"]["pageProps"]["gameData"]["values"]))
    
    levelsCategories = list(filter(filterCategoriesByPerLevelTrue, categories))
    fullGameCategories = list(filter(filterCategoriesByPerLevelFalse, categories))

    # Root node
    root = Node("root")

    # Full game nodes
    fullGameNode = Node("Full Game", parent=root)
    for category in fullGameCategories:
        Node(category["name"], id=category["id"], url=category["url"], parent=fullGameNode)

    # Levels nodes
    for level in levels:
        levelNode = Node(level["name"], id=level["id"], url=level["url"], parent=root)
        for category in levelsCategories:
            categoryNode = Node(category["name"], id=category["id"], url=category["url"], parent=levelNode)
            filteredVariables = list(filter(partial(filterVariablesForGivenLevelAndCategory, levelNode.get_attr("id"), categoryNode.get_attr("id")), variables))
            for variable in filteredVariables:
                variableNode = Node(variable["name"], id=variable["id"], url=variable["url"], parent=categoryNode)
                filteredValues = list(filter(partial(filterValuesForGivenVariable, variableNode.get_attr("id")), values))
                for value in filteredValues:
                    Node(value["name"], id=value["id"], url=value["url"], parent=variableNode)
                
    
    
    graph = tree_to_dot(root)
    graph.write_png("tree.png", encoding="utf-8")




jsonDict = json.load(open(f"experimental_data/TERRARIA.json", 'r', encoding='utf-8'))
getAllLinks(jsonDict)

                             ┌─        Moon Lord
                             ├─        All Bosses
        ┌─    Full Game     ─┼─ All Pre-Hardmode Bosses
        │                    ├─       Night's Edge
        │                    └─          Co-op
        │                                                                    ┌─    Random
        │                                                ┌─      Seeds      ─┤
        │                                                │                   └─    Seeded
        │                    ┌─         1 Player        ─┤                   ┌─   1.4 NMA
        │                    │                           │                   ├─ 1.4 Glitched
        │                    │                           └─ Patch/Game mode ─┼─   Journey
        │                    │                                               ├─ 1.3 Glitched
        ├─    King Slime    ─┤                                               └─   1.3 NMA
        │                    │  